# 01 — Data Collection & First Similarity Engine

This notebook builds the first working version of the **Champions League-Level Player Discovery** project.

The objective is to identify **non-Champions League strikers** with statistical profiles similar to strikers from Champions League clubs.

Current scope:
- Top 5 European leagues, 2025/2026 season
- Strikers only (`FW`)
- Role-specific per90 feature engineering
- Nearest Neighbors similarity model
- Example scouting report based on **Julián Álvarez**

In [ ]:
# Cell 1 — Import required libraries
# pandas and numpy are used for data handling.
# scikit-learn is used for feature scaling and similarity modeling.

import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [ ]:
# Cell 2 — Load raw player dataset
# The dataset is stored locally in the data/raw folder.
# It contains player statistics from Europe's top 5 leagues for the 2025/2026 season.

DATA_PATH = "../data/raw/players_data_light-2025_2026.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# Cell 3 — Basic dataset inspection
# This helps verify the size of the dataset and the available columns.

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Cell 4 — Inspect player position distribution
# This is important because the scouting engine should compare players by role.

df["Pos"].value_counts()

## Champions League benchmark definition

The project needs a benchmark group: clubs that participated in the 2025/2026 UEFA Champions League.

Players from these clubs are treated as the **UCL-level reference group**.  
Players from other clubs are treated as potential **non-UCL scouting targets**.

In [ ]:
# Cell 5 — Define Champions League clubs available in the dataset
# Club names must match exactly the names used in the dataset.

ucl_teams = [
    "Arsenal",
    "Athletic Club",
    "Atlético Madrid",
    "Atalanta",
    "Barcelona",
    "Bayern Munich",
    "Chelsea",
    "Dortmund",
    "Eintracht Frankfurt",
    "Inter",
    "Juventus",
    "Leverkusen",
    "Liverpool",
    "Manchester City",
    "Marseille",
    "Monaco",
    "Napoli",
    "Newcastle United",
    "Paris Saint-Germain",
    "Real Madrid",
    "Tottenham Hotspur",
    "Villarreal"
]

# Create a boolean flag identifying whether each player belongs to a UCL club.
df["is_ucl_team"] = df["Squad"].isin(ucl_teams)

df[["Player", "Squad", "is_ucl_team"]].head()

In [ ]:
# Cell 6 — Validate UCL flag on selected clubs
# This sanity check helps avoid wrong club classification.

df[df["Squad"].isin(["Atlético Madrid", "Milan", "Atalanta", "Athletic Club"])][
    ["Player", "Squad", "is_ucl_team"]
].drop_duplicates("Squad")

## Reliability filtering

Raw football data can be misleading when players have very few minutes.

To reduce noise, the first version of the model applies minimum sample thresholds:
- at least 15 appearances
- at least 900 minutes played

These thresholds can be adjusted in future versions.

In [ ]:
# Cell 7 — Apply minimum sample thresholds
# This improves the reliability of per90 statistics.

MIN_APPEARANCES = 15
MIN_MINUTES = 900

df_filtered = df[
    (df["MP"] >= MIN_APPEARANCES) &
    (df["Min"] >= MIN_MINUTES)
].copy()

print("Original dataset:", df.shape)
print("Filtered dataset:", df_filtered.shape)

## Striker subset

The first version of the engine focuses on strikers (`FW`).

Future versions can extend this logic to:
- wingers
- midfielders
- defensive midfielders
- fullbacks
- centre-backs
- goalkeepers

In [ ]:
# Cell 8 — Create striker subset
# We start with pure forwards only: Pos == "FW".

strikers = df_filtered[df_filtered["Pos"] == "FW"].copy()

print("Number of eligible strikers:", strikers.shape[0])
strikers.head()

In [ ]:
# Cell 9 — Select base striker statistics
# These are the raw attacking features used to build derived per90 metrics.

striker_features = [
    "Gls",
    "Ast",
    "G+A",
    "Sh",
    "SoT",
    "G/Sh"
]

striker_data = strikers[
    [
        "Player",
        "Squad",
        "Comp",
        "MP",
        "Min",
        "90s",
        "is_ucl_team"
    ] + striker_features
].copy()

striker_data.head()

## Feature engineering

Raw totals can be biased by playing time.

For scouting similarity, per90 metrics are more useful because they compare player output at a similar time scale.

Current engineered striker features:
- Goals per90
- Assists per90
- Shots per90
- Shots on target per90
- Goals per shot

In [ ]:
# Cell 10 — Create per90 striker features
# These features compare players independently from total minutes played.

striker_data["Gls_per90"] = striker_data["Gls"] / striker_data["90s"]
striker_data["Ast_per90"] = striker_data["Ast"] / striker_data["90s"]
striker_data["Sh_per90"] = striker_data["Sh"] / striker_data["90s"]
striker_data["SoT_per90"] = striker_data["SoT"] / striker_data["90s"]

advanced_striker_features = [
    "Gls_per90",
    "Ast_per90",
    "Sh_per90",
    "SoT_per90",
    "G/Sh"
]

striker_data = striker_data.reset_index(drop=True)

striker_data.head()

## Similarity model

The model uses:
- `StandardScaler` to normalize features
- `NearestNeighbors` to find statistically similar players
- Euclidean distance as the similarity distance metric

Lower distance means more similar.  
A derived `similarity_score` is added for easier interpretation.

In [ ]:
# Cell 11 — Scale features and train Nearest Neighbors model
# Scaling prevents high-volume statistics from dominating the distance calculation.

X = striker_data[advanced_striker_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = NearestNeighbors(
    n_neighbors=10,
    metric="euclidean"
)

model.fit(X_scaled)

In [ ]:
# Cell 12 — Define reusable scouting function
# Given a UCL benchmark striker, the function returns similar non-UCL strikers.

def find_similar_non_ucl_players(player_name, n_neighbors=15):
    """
    Find non-UCL strikers with statistical profiles similar to a selected striker.

    Parameters
    ----------
    player_name : str
        Exact player name as written in the dataset.
    n_neighbors : int
        Number of nearest neighbors to retrieve before filtering non-UCL players.

    Returns
    -------
    pandas.DataFrame
        Non-UCL players ranked by similarity score.
    """

    player_match = striker_data[striker_data["Player"] == player_name]

    if player_match.empty:
        return f"Player '{player_name}' not found. Check spelling, accents, or position filter."

    player_index = player_match.index[0]

    distances, indices = model.kneighbors(
        [X_scaled[player_index]],
        n_neighbors=n_neighbors
    )

    similar_players = striker_data.iloc[indices[0]].copy()
    similar_players["distance"] = distances[0]

    # Convert distance into a more readable score.
    # Higher score = more similar profile.
    similar_players["similarity_score"] = 100 / (1 + similar_players["distance"])

    hidden_targets = similar_players[
        similar_players["is_ucl_team"] == False
    ].copy()

    return hidden_targets.sort_values("similarity_score", ascending=False)

## Example scouting report

Benchmark player: **Julián Álvarez**  
Club: **Atlético Madrid**  
Status: **Champions League club**

The engine searches for non-UCL strikers with similar statistical profiles.

In [ ]:
# Cell 13 — Generate example scouting report

target_player = "Julián Álvarez"

results = find_similar_non_ucl_players(
    target_player,
    n_neighbors=15
)

results

In [ ]:
# Cell 14 — Export scouting report to processed data folder
# This creates a reusable CSV output for the repository.

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/julian_alvarez_similar_non_ucl_strikers.csv"

results.to_csv(output_path, index=False)

print(f"Scouting report saved to: {output_path}")

In [ ]:
# Cell 15 — Compact report view
# A cleaner output for quick interpretation.

results[
    [
        "Player",
        "Squad",
        "Comp",
        "MP",
        "Min",
        "90s",
        "Gls",
        "Ast",
        "G+A",
        "Gls_per90",
        "Ast_per90",
        "Sh_per90",
        "SoT_per90",
        "similarity_score"
    ]
].round(3)

## Next steps

Possible improvements:
1. Extend the engine to additional roles.
2. Add advanced FBref metrics such as xG, xA, progressive passes, and progressive carries.
3. Add market value and age to identify cost-efficient transfer targets.
4. Create radar charts for player comparison.
5. Build a Streamlit dashboard to make the scouting engine interactive.